# Capitals multi-step retrieval: 10,000-prompt TopK extension

This post-report extension trains new TopK-128 SAEs on a relation-balanced geography corpus: 5,000 prompts ask for the capital associated with a non-capital city, and 5,000 ask for the country or state containing that city. The causal test asks whether a compact feature panel selectively supports the multi-step `city -> country -> capital` relation on disjoint countries. All artifacts use a new namespace and cannot overwrite the original capitals experiments.

## 1. Mount Drive, fetch the repository, and install it

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shlex, subprocess, sys
from pathlib import Path

def run_cmd(command):
    command = [str(part) for part in command]
    print('$', shlex.join(command), flush=True)
    environment = os.environ.copy()
    environment['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=environment)
    if process.stdout is None:
        raise RuntimeError('Could not capture child-process output')
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

repo_url = 'https://github.com/evey-dev/test_run.git'
checkout = Path('/content/test_run')
if (checkout / '.git').is_dir():
    run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
else:
    if checkout.exists() and any(checkout.iterdir()):
        checkout = Path('/content/test_run_github')
    if (checkout / '.git').is_dir():
        run_cmd(['git', '-C', checkout, 'pull', '--ff-only'])
    else:
        run_cmd(['git', 'clone', '--depth', '1', repo_url, checkout])
os.chdir(checkout)
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.51,<5', 'accelerate', 'pandas<2.4', 'pyyaml', 'tqdm'])
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
print('Repository root:', Path.cwd())

Mounted at /content/drive
$ git clone --depth 1 https://github.com/evey-dev/test_run.git /content/test_run
Cloning into '/content/test_run'...
$ /usr/bin/python3 -m pip install -q 'transformers>=4.51,<5' accelerate 'pandas<2.4' pyyaml tqdm
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 150.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
$ /usr/bin/python3 -m pip install -q -e .
Repository root: /content/test_run


## 2. Fixed design and persistent artifact paths

In [3]:
import csv, json, re, shutil
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display

ROOT = Path.cwd().resolve()
LAYERS = [4, 8, 12, 16, 20, 24, 28]
DATA_ARG = 'data/capitals_large_10000.csv'
CONFIG_ARG = 'configs/sae_capitals_large_10000_topk128_config.yaml'
ACTIVATION_DIR = ROOT / 'mechanistic_data_capitals_large_10000'
CHECKPOINT_DIR = ROOT / 'mechanistic_data/sae_checkpoints_capitals_large_10000_topk128'
LOCAL_OUTPUT = ROOT / 'outputs/capitals_large_data_test'
DRIVE_ROOT = Path('/content/drive/MyDrive/mphil-project')
DRIVE_EXPERIMENT = DRIVE_ROOT / 'mechanistic_data/capitals_large_data_test'
DRIVE_ACTIVATIONS = DRIVE_EXPERIMENT / 'activations'
DRIVE_CHECKPOINTS = DRIVE_EXPERIMENT / 'checkpoints_topk128'
DRIVE_OUTPUT = DRIVE_ROOT / 'outputs/capitals_large_data_test'
for path in (LOCAL_OUTPUT, DRIVE_EXPERIMENT, DRIVE_ACTIVATIONS, DRIVE_CHECKPOINTS, DRIVE_OUTPUT):
    path.mkdir(parents=True, exist_ok=True)

required = [Path('data/generate_large_capitals_dataset.py'), Path(CONFIG_ARG), Path('src/capitals_relation_feature_screen.py')]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Push the new experiment files to GitHub before running: ' + ', '.join(missing))
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing')
config = yaml.safe_load(Path(CONFIG_ARG).read_text())
assert config['activation_type'] == 'topk' and config['top_k'] == 128
assert config['layers'] == LAYERS and config['seed'] == 787 and config.get('save_latents') is False
print('GPU:', torch.cuda.get_device_name(0))
print('Drive experiment root:', DRIVE_EXPERIMENT)

GPU: NVIDIA A100-SXM4-80GB
Drive experiment root: /content/drive/MyDrive/mphil-project/mechanistic_data/capitals_large_data_test


## 3. Generate and audit the relation-balanced corpus

The exact prompts later used for discovery and confirmation must be absent from this SAE corpus. The graph entity, Jordan, is also excluded from both intervention splits.

In [4]:
run_cmd([sys.executable, 'data/generate_large_capitals_dataset.py', '--count', '10000', '--seed', '787', '--base-csv', 'data/capitals_data.csv', '--output', DATA_ARG])
corpus = pd.read_csv(DATA_ARG)
assert len(corpus) == 10_000 and corpus['sentence'].nunique() == 10_000
assert corpus.groupby('Relation').size().to_dict() == {'capital': 5000, 'location': 5000}
original = pd.read_csv('data/capitals_data.csv')
assert set(original['sentence']).issubset(set(corpus['sentence']))

from src.capitals_relation_feature_screen import candidate_cases
screen_candidates = candidate_cases(original.to_dict('records'), seed=4787, excluded_countries=['Jordan'])
screen_prompts = {row[key] for row in screen_candidates for key in ('capital_prompt', 'country_prompt')}
assert set(corpus['sentence']).isdisjoint(screen_prompts)
assert all(row['country'] != 'Jordan' for row in screen_candidates)
display(corpus.groupby(['Relation', 'Type']).size().unstack(fill_value=0))
display(corpus[['Relation', 'Location', 'ContextCity', 'sentence']].head())

$ /usr/bin/python3 data/generate_large_capitals_dataset.py --count 10000 --seed 787 --base-csv data/capitals_data.csv --output data/capitals_large_10000.csv
Saved 10000 unique prompts to data/capitals_large_10000.csv
Relation counts: {'capital': 5000, 'location': 5000}
Distinct locations: 186


Type,country,state
Relation,,
capital,4586,414
location,4632,368


,Relation,Location,ContextCity,sentence
0,location,Luxembourg,Esch-sur-Alzette,Fact: The country in which Esch-sur-Alzette is...
1,capital,Afghanistan,Kandahar,Question: Kandahar lies within a country. What...
2,capital,Australia,Coombs,Fact: The governmental seat of the country in ...
3,location,Iran,Gushalan,Fact: The political region containing Gushalan...
4,capital,Chile,San Gregorio,Question: San Gregorio lies within a country. ...


## 4. Capture final-token MLP activations at seven selected layers

In [5]:
activation_files = [f'activations_layer{layer}.npy' for layer in LAYERS] + ['train_indices.npy', 'val_indices.npy', 'train_val_indices_per_layer.npy', 'activation_metadata.csv']
def complete(directory, names):
    return all((directory / name).exists() for name in names)

if not complete(ACTIVATION_DIR, activation_files) and complete(DRIVE_ACTIVATIONS, activation_files):
    print('Restoring activation capture from Drive...')
    shutil.copytree(DRIVE_ACTIVATIONS, ACTIVATION_DIR, dirs_exist_ok=True)
if not complete(ACTIVATION_DIR, activation_files):
    run_cmd([sys.executable, '-m', 'src.capture_activations', '--model-config', 'configs/model_config.yaml', '--output-dir', str(ACTIVATION_DIR), '--prompt-csv', DATA_ARG, '--prompt-behaviour', 'capitals_relation_large_10000', '--layers', *map(str, LAYERS), '--seed', '787'])
    shutil.copytree(ACTIVATION_DIR, DRIVE_ACTIVATIONS, dirs_exist_ok=True)
else:
    print('Activation capture already complete; skipping capture.')
assert complete(ACTIVATION_DIR, activation_files)
train_indices = np.load(ACTIVATION_DIR / 'train_indices.npy')
val_indices = np.load(ACTIVATION_DIR / 'val_indices.npy')
assert len(train_indices) == 8000 and len(val_indices) == 2000
display(pd.DataFrame({'train': corpus.iloc[train_indices].groupby('Relation').size(), 'validation': corpus.iloc[val_indices].groupby('Relation').size()}))

$ /usr/bin/python3 -m src.capture_activations --model-config configs/model_config.yaml --output-dir /content/test_run/mechanistic_data_capitals_large_10000 --prompt-csv data/capitals_large_10000.csv --prompt-behaviour capitals_relation_large_10000 --layers 4 8 12 16 20 24 28 --seed 787
Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
2026-07-16 05:08:07.959962: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-16 05:08:08.035884: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other opera

,train,validation
Relation,,
capital,4003,997
location,3997,1003


## 5. Train or restore the 10k TopK-128 SAEs

In [6]:
run_cmd([sys.executable, '-m', 'src.train', '--config', CONFIG_ARG, '--drive-dir', str(DRIVE_CHECKPOINTS)])
checkpoint_files = [name for layer in LAYERS for name in (f'sae_layer{layer}.pt', f'sae_layer{layer}_metadata.json')]
assert complete(CHECKPOINT_DIR, checkpoint_files)
print('Complete checkpoint layers:', LAYERS)
print('Dense latent arrays written:', len(list(CHECKPOINT_DIR.glob('latents_layer*.npy'))))

$ /usr/bin/python3 -m src.train --config configs/sae_capitals_large_10000_topk128_config.yaml --drive-dir /content/drive/MyDrive/mphil-project/mechanistic_data/capitals_large_data_test/checkpoints_topk128
Using CUDA device: NVIDIA A100-SXM4-80GB

=== Training SAE for layer 4 ===
Layer 4 activation scaling factor (raw mean L2 / sqrt(d_in)): 0.1547
SAE activation: topk (k=128) | decoder unit-norm constraint: True
Epoch 00 | train_loss=0.0861 | val_loss=0.0269 | mse=0.0269 | l1=0.0146
Epoch 01 | train_loss=0.0185 | val_loss=0.0147 | mse=0.0147 | l1=0.0127
Epoch 02 | train_loss=0.0132 | val_loss=0.0118 | mse=0.0118 | l1=0.0138
Epoch 03 | train_loss=0.0106 | val_loss=0.0101 | mse=0.0101 | l1=0.0140
Epoch 04 | train_loss=0.0099 | val_loss=0.0096 | mse=0.0096 | l1=0.0149
Epoch 05 | train_loss=0.0094 | val_loss=0.0095 | mse=0.0095 | l1=0.0148
Epoch 06 | train_loss=0.0084 | val_loss=0.0084 | mse=0.0084 | l1=0.0152
Epoch 07 | train_loss=0.0075 | val_loss=0.0076 | mse=0.0076 | l1=0.0152
Epoch 08 

## 6. Measure held-out reconstruction and sparsity

In [7]:
diagnostic_json = LOCAL_OUTPUT / 'capitals_large_10000_topk128_diagnostics.json'
diagnostic_csv = LOCAL_OUTPUT / 'capitals_large_10000_topk128_diagnostics.csv'
if (DRIVE_OUTPUT / diagnostic_json.name).exists() and (DRIVE_OUTPUT / diagnostic_csv.name).exists():
    shutil.copy2(DRIVE_OUTPUT / diagnostic_json.name, diagnostic_json)
    shutil.copy2(DRIVE_OUTPUT / diagnostic_csv.name, diagnostic_csv)
else:
    run_cmd([sys.executable, '-m', 'src.sae_diagnostics', '--config', CONFIG_ARG, '--label', 'capitals_large_10000_topk128', '--device', 'cuda', '--output-json', str(diagnostic_json), '--output-csv', str(diagnostic_csv)])
    shutil.copy2(diagnostic_json, DRIVE_OUTPUT / diagnostic_json.name)
    shutil.copy2(diagnostic_csv, DRIVE_OUTPUT / diagnostic_csv.name)
display(pd.read_csv(diagnostic_csv)[['layer', 'validation_fraction_variance_explained', 'validation_mean_relative_l2_error', 'validation_mean_l0', 'combined_dead_feature_fraction']])

$ /usr/bin/python3 -m src.sae_diagnostics --config configs/sae_capitals_large_10000_topk128_config.yaml --label capitals_large_10000_topk128 --device cuda --output-json /content/test_run/outputs/capitals_large_data_test/capitals_large_10000_topk128_diagnostics.json --output-csv /content/test_run/outputs/capitals_large_data_test/capitals_large_10000_topk128_diagnostics.csv
Evaluating capitals_large_10000_topk128 SAE at layer 4 on cuda...
Evaluating capitals_large_10000_topk128 SAE at layer 8 on cuda...
Evaluating capitals_large_10000_topk128 SAE at layer 12 on cuda...
Evaluating capitals_large_10000_topk128 SAE at layer 16 on cuda...
Evaluating capitals_large_10000_topk128 SAE at layer 20 on cuda...
Evaluating capitals_large_10000_topk128 SAE at layer 24 on cuda...
Evaluating capitals_large_10000_topk128 SAE at layer 28 on cuda...

SAE diagnostic summary (validation split)
domain      layer      FVE   rel_L2   mean_L0   dead_all
capitals_large_10000_topk128     4    0.989    0.067      

,layer,validation_fraction_variance_explained,validation_mean_relative_l2_error,validation_mean_l0,combined_dead_feature_fraction
0,4,0.989389,0.067218,80.5505,0.928833
1,8,0.976183,0.121931,71.2370,0.903687
2,12,0.964941,0.138956,113.6750,0.906860
3,16,0.959639,0.155139,124.2790,0.912354
4,20,0.966486,0.141305,120.5055,0.911743
5,24,0.954175,0.148890,125.4685,0.914795
6,28,0.954055,0.165345,126.2545,0.915771


## 7. Build the Amman-over-Jordan contrast graph

The graph prompt requires city-to-country-to-capital retrieval. Positive graph features are only candidates; the disjoint-country intervention screen supplies the causal test.

In [8]:
GRAPH_STEM = 'capitals_large_10000_topk128_amman_over_jordan_graph'
graph_paths = {suffix: LOCAL_OUTPUT / f'{GRAPH_STEM}.{suffix}' for suffix in ('json', 'html', 'md')}
drive_graph_paths = {suffix: DRIVE_OUTPUT / path.name for suffix, path in graph_paths.items()}
if all(path.exists() for path in drive_graph_paths.values()):
    for suffix in graph_paths:
        shutil.copy2(drive_graph_paths[suffix], graph_paths[suffix])
else:
    run_cmd([sys.executable, '-m', 'src.attribution_graph', '--prompt', 'Fact: The capital of the country containing Zarqa is named', '--target', 'Amman', '--contrast-target', 'Jordan', '--layers', *map(str, LAYERS), '--sae-config', CONFIG_ARG, '--top-k-nodes', '25', '--top-k-edges', '35', '--output-json', str(graph_paths['json']), '--output-html', str(graph_paths['html']), '--output-mermaid', str(graph_paths['md'])])
    for suffix in graph_paths:
        shutil.copy2(graph_paths[suffix], drive_graph_paths[suffix])
graph = json.loads(graph_paths['json'].read_text())
positive_features = [node for node in graph['nodes'] if re.fullmatch(r'layer_\d+_feature_\d+', str(node.get('id', ''))) and float(node.get('attribution', 0.0)) > 0]
print('Objective:', graph['attribution_objective'])
print('Objective value:', graph['attribution_objective_value'])
print('Positive graph feature candidates:', len(positive_features))

$ /usr/bin/python3 -m src.attribution_graph --prompt 'Fact: The capital of the country containing Zarqa is named' --target Amman --contrast-target Jordan --layers 4 8 12 16 20 24 28 --sae-config configs/sae_capitals_large_10000_topk128_config.yaml --top-k-nodes 25 --top-k-edges 35 --output-json /content/test_run/outputs/capitals_large_data_test/capitals_large_10000_topk128_amman_over_jordan_graph.json --output-html /content/test_run/outputs/capitals_large_data_test/capitals_large_10000_topk128_amman_over_jordan_graph.html --output-mermaid /content/test_run/outputs/capitals_large_data_test/capitals_large_10000_topk128_amman_over_jordan_graph.md
Loading model and tokenizer...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
2026-07-16 05:24:27.671098: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from differen

## 8. Discover and confirm a capital-relation panel

Features are ranked on eight discovery countries and frozen before testing on sixteen disjoint confirmation countries. The matched inverse prompt asks for the country itself. The primary estimand is the inhibition-induced change in `logit(capital) - logit(country)` on the capital prompt minus the same change on the inverse prompt.

In [9]:
drive_screen = DRIVE_OUTPUT / 'capitals_large_10000_topk128_relation_screen.json'
run_cmd([sys.executable, '-m', 'src.capitals_relation_feature_screen', '--sae-config', CONFIG_ARG, '--capitals-csv', 'data/capitals_data.csv', '--sae-corpus-csv', DATA_ARG, '--graph', str(graph_paths['json']), '--discovery-cases', '8', '--confirmation-cases', '16', '--panel-sizes', '1', '3', '5', '10', '20', '--primary-panel-size', '3', '--random-panels', '5', '--output', str(drive_screen)])
screen_path = LOCAL_OUTPUT / drive_screen.name
shutil.copy2(drive_screen, screen_path)
screen = json.loads(screen_path.read_text())
primary = screen['confirmation']['primary_result']
print('Status:', screen['status'])
print('Frozen Top-3:', screen['discovery']['frozen_feature_order'][:3])
print('Confirmatory rule passed:', primary['supports_capital_relation_selectivity_under_predeclared_rule'])
display(pd.Series(primary['summary']).to_frame('value'))
if primary['supports_capital_relation_selectivity_under_predeclared_rule']:
    print('Safe presentation claim: a compact panel selectively supports multi-step capital retrieval on held-out countries.')
else:
    print('Safe presentation claim: the 10k relation-balanced SAEs did not confirm a compact capital-relation panel.')

$ /usr/bin/python3 -m src.capitals_relation_feature_screen --sae-config configs/sae_capitals_large_10000_topk128_config.yaml --capitals-csv data/capitals_data.csv --sae-corpus-csv data/capitals_large_10000.csv --graph /content/test_run/outputs/capitals_large_data_test/capitals_large_10000_topk128_amman_over_jordan_graph.json --discovery-cases 8 --confirmation-cases 16 --panel-sizes 1 3 5 10 20 --primary-panel-size 3 --random-panels 5 --output /content/drive/MyDrive/mphil-project/outputs/capitals_large_data_test/capitals_large_10000_topk128_relation_screen.json
Loading model and capitals SAEs...
Local model path './models/Qwen3-4B-Instruct' not found. Falling back to Hugging Face Hub: Qwen/Qwen3-4B-Instruct-2507
2026-07-16 05:24:57.757110: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_O

,value
eligible_cases,16
mean_capital_prompt_delta,-1.257812
bootstrap_95_ci_mean_capital_prompt_delta,"[-1.6015625, -0.90625]"
mean_inverse_country_prompt_delta,-0.15625
bootstrap_95_ci_mean_inverse_country_prompt_delta,"[-0.224609375, -0.08984375]"
mean_relation_specific_difference,-1.101562
bootstrap_95_ci_mean_relation_specific_difference,"[-1.449267578125, -0.7499511718750007]"
fraction_capital_delta_negative,0.875
fraction_relation_specific_difference_negative,0.9375
capital_to_country_flip_fraction,0.0


Safe presentation claim: a compact panel selectively supports multi-step capital retrieval on held-out countries.


## 9. Save provenance and verify persistent artifacts

In [10]:
for path in LOCAL_OUTPUT.iterdir():
    if path.is_file():
        shutil.copy2(path, DRIVE_OUTPUT / path.name)
provenance = DRIVE_EXPERIMENT / 'provenance'
provenance.mkdir(parents=True, exist_ok=True)
shutil.copy2(DATA_ARG, provenance / Path(DATA_ARG).name)
shutil.copy2(CONFIG_ARG, provenance / Path(CONFIG_ARG).name)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() if (ROOT / '.git').exists() else None
manifest = {'experiment': 'capitals_large_10000_topk128_relation', 'git_commit': commit, 'gpu': torch.cuda.get_device_name(0), 'seed': 787, 'layers': LAYERS, 'corpus_rows': len(corpus), 'relation_counts': corpus.groupby('Relation').size().to_dict(), 'train_rows': len(train_indices), 'validation_rows': len(val_indices), 'config': CONFIG_ARG, 'dense_latent_export': False, 'drive_output': str(DRIVE_OUTPUT), 'drive_checkpoints': str(DRIVE_CHECKPOINTS), 'drive_activations': str(DRIVE_ACTIVATIONS)}
manifest_path = LOCAL_OUTPUT / 'experiment_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
shutil.copy2(manifest_path, DRIVE_OUTPUT / manifest_path.name)
print(json.dumps(manifest, indent=2))
print('All persistent outputs are under:', DRIVE_OUTPUT)

{
  "experiment": "capitals_large_10000_topk128_relation",
  "git_commit": "406afd377810d40baa50f2bb45826925f64f245d",
  "gpu": "NVIDIA A100-SXM4-80GB",
  "seed": 787,
  "layers": [
    4,
    8,
    12,
    16,
    20,
    24,
    28
  ],
  "corpus_rows": 10000,
  "relation_counts": {
    "capital": 5000,
    "location": 5000
  },
  "train_rows": 8000,
  "validation_rows": 2000,
  "config": "configs/sae_capitals_large_10000_topk128_config.yaml",
  "dense_latent_export": false,
  "drive_output": "/content/drive/MyDrive/mphil-project/outputs/capitals_large_data_test",
  "drive_checkpoints": "/content/drive/MyDrive/mphil-project/mechanistic_data/capitals_large_data_test/checkpoints_topk128",
  "drive_activations": "/content/drive/MyDrive/mphil-project/mechanistic_data/capitals_large_data_test/activations"
}
All persistent outputs are under: /content/drive/MyDrive/mphil-project/outputs/capitals_large_data_test
